# Notebook 1: GELU Polynomial Approximation

The first concrete problem in private ViT inference: how do we evaluate GELU in FHE?

CKKS only supports polynomial operations. We need to approximate GELU as a polynomial.
The key trade-off: **degree vs multiplicative depth vs accuracy**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../')
from utils.poly_approx import chebyshev_approx, minimax_approx, gelu, approx_error, poly_depth

## What is GELU?

$$\text{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right]$$

Approximated as:
$$\text{GELU}(x) \approx 0.5x\left[1 + \tanh\left(\sqrt{\frac{2}{\pi}}(x + 0.044715x^3)\right)\right]$$

Neither form is a polynomial. We need to find $p(x)$ of bounded degree such that $|\text{GELU}(x) - p(x)|$ is small on the domain of interest.

In [ ]:
domain = (-6, 6)
x = np.linspace(*domain, 1000)

plt.figure(figsize=(10, 4))
plt.plot(x, gelu(x), 'b-', linewidth=2, label='True GELU')
plt.xlabel('x')
plt.ylabel('GELU(x)')
plt.title('GELU Activation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Chebyshev Approximation

We use Chebyshev polynomials as initial approximations. For a function $f$ on $[-1, 1]$:

$$f(x) \approx \sum_{k=0}^{n} c_k T_k(x)$$

We transform to our domain $[a, b]$ and convert back to the monomial basis.

In [ ]:
degrees = [4, 7, 15, 27]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, degree in zip(axes, degrees):
    coeffs = chebyshev_approx(gelu, degree, domain)
    y_approx = sum(c * x**j for j, c in enumerate(coeffs))
    metrics = approx_error(gelu, coeffs, domain)
    
    ax.plot(x, gelu(x), 'b-', alpha=0.7, label='True')
    ax.plot(x, y_approx, 'r--', label=f'Deg {degree}')
    ax.set_title(f'Degree {degree} (depth {poly_depth(degree)})\nMax err: {metrics["max_abs_error"]:.2e}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('GELU Polynomial Approximations')
plt.tight_layout()
plt.show()

## Depth vs Error Trade-off

Key question: what degree do we need for <1% accuracy loss in the final model?

In [ ]:
all_degrees = list(range(2, 30))
errors = []
depths = []

for d in all_degrees:
    coeffs = chebyshev_approx(gelu, d, domain)
    m = approx_error(gelu, coeffs, domain)
    errors.append(m['max_abs_error'])
    depths.append(poly_depth(d))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.semilogy(all_degrees, errors, 'o-', color='steelblue')
ax1.axhline(y=0.01, color='red', linestyle='--', label='0.01 threshold')
ax1.set_xlabel('Polynomial Degree')
ax1.set_ylabel('Max Absolute Error (log scale)')
ax1.set_title('Approximation Error vs Degree')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.scatter(depths, errors, c=all_degrees, cmap='viridis', s=60)
ax2.set_xlabel('Multiplicative Depth (levels consumed)')
ax2.set_ylabel('Max Absolute Error')
ax2.set_title('Error vs Depth (Pareto Frontier)')
ax2.set_yscale('log')
plt.colorbar(ax2.collections[0], ax=ax2, label='Degree')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Which degree first hits error < 0.01?
threshold = 0.01
for d, e in zip(all_degrees, errors):
    if e < threshold:
        print(f'First degree with max error < {threshold}: degree {d} (depth {poly_depth(d)})')
        break

## Exercise: Minimax vs Chebyshev

Chebyshev is fast but not optimal. The minimax (Remez) algorithm finds the polynomial that **minimizes the maximum error** — which is what we actually want for worst-case guarantees.

Try computing the minimax approximation for degree 7 and compare to Chebyshev.

In [ ]:
# TODO: Compare minimax vs Chebyshev at degree 7
# coeffs_cheb = chebyshev_approx(gelu, 7, domain)
# coeffs_mini = minimax_approx(gelu, 7, domain)  # slower but better
# 
# Compare their max errors and plot both